# Idealista API: raw -> preprocess consolidado

Este cuaderno sustituye la limpieza automatica en codigo.

Objetivo:
- leer todos los runs raw de una operacion (`rent` o `sale`)
- unir todos los JSON encontrados
- eliminar duplicados priorizando los runs mas recientes
- exportar un unico CSV a `data/raw/idealistaAPI/preprocess/total_*_cantabria.csv`
- mostrar estadisticas por run: totales, unicos y duplicados


In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "src").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("No se encontro la raiz del proyecto (carpeta 'src').")

RAW_BASE = PROJECT_ROOT / "data" / "raw" / "idealistaAPI" / "raw"
PREPROCESS_BASE = PROJECT_ROOT / "data" / "raw" / "idealistaAPI" / "preprocess"


## Operacion a consolidar

Cambia `OPERATION` a `"sale"` o `"rent"`.


In [ ]:
OPERATION = "sale"

OUTPUT_NAME_MAP = {
    "sale": "total_sales_cantabria.csv",
    "rent": "total_rent_cantabria.csv",
}

SUMMARY_NAME_MAP = {
    "sale": "total_sales_cantabria_summary.json",
    "rent": "total_rent_cantabria_summary.json",
}

if OPERATION not in OUTPUT_NAME_MAP:
    raise ValueError(f"Operacion no soportada: {OPERATION}")

OUTPUT_CSV = PREPROCESS_BASE / OUTPUT_NAME_MAP[OPERATION]
OUTPUT_SUMMARY = PREPROCESS_BASE / SUMMARY_NAME_MAP[OPERATION]

print(f"Operacion seleccionada: {OPERATION}")
print(f"OUTPUT_CSV: {OUTPUT_CSV}")


## Runs raw detectados para la operacion


In [ ]:
operation_runs = sorted(
    [path for path in RAW_BASE.glob(f"{OPERATION}_homes_run_*") if path.is_dir()],
    reverse=True,
)

if not operation_runs:
    raise FileNotFoundError(f"No se encontraron runs raw para operation='{OPERATION}' en {RAW_BASE}")

runs_df = pd.DataFrame(
    [
        {
            "run_name": path.name,
            "json_files": len(list(path.glob("*.json"))),
            "has_manifest": (path / "manifest.json").exists(),
        }
        for path in operation_runs
    ]
)

runs_df


## Funciones de carga y deduplicado


In [ ]:
def extract_all_elements(input_dir: Path) -> list[dict]:
    rows: list[dict] = []
    for json_path in sorted(input_dir.glob("*.json")):
        if json_path.name.lower() == "manifest.json":
            continue
        try:
            payload = json.loads(json_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        element_list = payload.get("elementList")
        if isinstance(element_list, list):
            rows.extend(item for item in element_list if isinstance(item, dict))
    return rows


def build_property_dedupe_key(df: pd.DataFrame) -> pd.Series:
    property_code = df.get("propertyCode", pd.Series([""] * len(df), index=df.index)).fillna("").astype(str).str.strip()
    fallback = (
        df.get("price", pd.Series([""] * len(df), index=df.index)).fillna("").astype(str)
        + "|"
        + df.get("size", pd.Series([""] * len(df), index=df.index)).fillna("").astype(str)
        + "|"
        + df.get("latitude", pd.Series([""] * len(df), index=df.index)).fillna("").astype(str)
        + "|"
        + df.get("longitude", pd.Series([""] * len(df), index=df.index)).fillna("").astype(str)
        + "|"
        + df.get("address", df.get("streetName", pd.Series([""] * len(df), index=df.index))).fillna("").astype(str).str.strip().str.lower()
    )
    return property_code.where(property_code != "", fallback)


def load_operation_runs_as_dataframe(run_dirs: list[Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    frames: list[pd.DataFrame] = []
    run_stats: list[dict] = []

    for run_dir in run_dirs:
        rows = extract_all_elements(run_dir)
        if not rows:
            print(f"{run_dir.name}: total=0 unicos=0 duplicados=0")
            run_stats.append(
                {
                    "run_name": run_dir.name,
                    "rows_total": 0,
                    "rows_unique": 0,
                    "rows_duplicated": 0,
                }
            )
            continue

        df_run = pd.json_normalize(rows)
        df_run["_dedupe_key"] = build_property_dedupe_key(df_run)

        rows_total = int(len(df_run))
        rows_duplicated = int(df_run.duplicated(subset="_dedupe_key").sum())
        rows_unique = rows_total - rows_duplicated

        print(f"{run_dir.name}: total={rows_total} unicos={rows_unique} duplicados={rows_duplicated}")

        run_stats.append(
            {
                "run_name": run_dir.name,
                "rows_total": rows_total,
                "rows_unique": rows_unique,
                "rows_duplicated": rows_duplicated,
            }
        )

        df_run = df_run.drop(columns=["_dedupe_key"])
        df_run["source_run"] = run_dir.name
        frames.append(df_run)

    if not frames:
        raise ValueError(f"No se encontraron anuncios validos para operation='{OPERATION}'")

    combined_df = pd.concat(frames, ignore_index=True, sort=False)
    run_stats_df = pd.DataFrame(run_stats)
    return combined_df, run_stats_df


## Carga consolidada de la operacion


In [ ]:
df_raw, run_stats_df = load_operation_runs_as_dataframe(operation_runs)

print(f"Runs incluidos: {len(operation_runs)}")
print(f"Filas raw totales: {len(df_raw)}")
print(f"Columnas: {len(df_raw.columns)}")

df_raw.head()


## Resumen por run


In [ ]:
run_stats_df


## Deduplicado consolidado


In [ ]:
df_raw["_dedupe_key"] = build_property_dedupe_key(df_raw)
duplicate_rows = int(df_raw.duplicated(subset="_dedupe_key").sum())
df_preprocess = df_raw.drop_duplicates(subset="_dedupe_key", keep="first").drop(columns=["_dedupe_key"]).copy()

print(f"Filas raw totales: {len(df_raw)}")
print(f"Duplicados detectados en consolidacion: {duplicate_rows}")
print(f"Filas preprocess finales: {len(df_preprocess)}")

df_preprocess.head()


## Export del CSV total


In [ ]:
PREPROCESS_BASE.mkdir(parents=True, exist_ok=True)
df_preprocess.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

summary = {
    "operation": OPERATION,
    "raw_base": str(RAW_BASE),
    "runs_included": [run_dir.name for run_dir in operation_runs],
    "output_csv": str(OUTPUT_CSV),
    "rows_raw": int(len(df_raw)),
    "rows_output": int(len(df_preprocess)),
    "duplicates_removed": int(duplicate_rows),
    "priority_rule": "latest_run_first",
    "run_stats": run_stats_df.to_dict(orient="records"),
}

OUTPUT_SUMMARY.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"CSV exportado en: {OUTPUT_CSV.resolve()}")
summary
